> **Local safe adaptation.** The 20 exercise cells below come from [srush/Autodiff-Puzzles](https://github.com/srush/Autodiff-Puzzles) at commit `ae4586b`. Generated outputs and the runtime GitHub package installer were removed; `checker.py` supplies the JAX correctness check locally.

# Autodiff Puzzles
- by [Sasha Rush](http://rush-nlp.com) - [srush_nlp](https://twitter.com/srush_nlp)


This is a set of puzzles  about getting better at autodifferentiation
It's a follow up to the [Tensor Puzzles](https://github.com/srush/Tensor-Puzzles).



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srush/Autodiff-Puzzles/blob/main/autodiff_puzzlers.ipynb)



These puzzles are about grokking autodiff.
I give you a basic tensor-to-tensor function, and your goal is to give me the Jacobian.
If a function takes input a tensor of shape $I$ and outputs a tensor of shape $O$ the
Jacobian is a tensor of shape $(O, I)$  that contains each derivative $\frac{df(x)_o}{dx_i}$.
My recommendation is to work them out first with math and then think about how to write
this as a tensor function. It gets really hard.

## Rules and Tips

* Every answer is 1 line of 80-column code. Many of them can be done with 20 characters or so.
* Everything in these puzzles should be done basic operators, nothing fancy. You cannot call np.array, etc.
* Recall the basic multivariate calculus identities.

The visualization will show you $x$ and $f(x)$ as heat maps.
The spotlights show you what your current Jacobian looks like.

In [ ]:
# Local library code: no runtime downloads or package installation.
from checker import Array, Shaped, check, np


### Problem 1: Id

Warmup: $f(x_0) = [x_0]$

In [ ]:

Is = np.arange(1)


def f(x: Shaped[Array, "1"]) -> Shaped[Array, "1"]:
    return 2 * x


def jac(x: Shaped[Array, "1"]) -> Shaped[Array, "1 1"]:
    return 0 *  Is[None, :]  # fill in


check(f, jac, in_shape=Is.shape)

### Problem 2: Cosine

Warmup: $f(x_0) = [\cos(x_0)]$

In [ ]:
def f(x: Shaped[Array, "1"]) -> Shaped[Array, "1"]:
    return np.cos(x)


def jac(x: Shaped[Array, "1"]) -> Shaped[Array, "1 1"]:
    return 0. * Is[None, :]  # fill in


check(f, jac, in_shape=1)

### Problem 3: Mean

$f(x_0, x_1, \ldots, x_{I-1}) = [(x_0 + x_1 + \ldots + x_{I-1}) / I]$

In [ ]:
I = 10
Is = np.arange(I)


def f(x: Shaped[Array, "I"]) -> Shaped[Array, "1"]:
    return np.mean(x, axis=0, keepdims=True)


def jac(x: Shaped[Array, "I"]) -> Shaped[Array, "1 I"]:
    return 0 * Is[None, :]  # fill in


check(f, jac, in_shape=Is.shape)

### Problem 4: Product

$f(x_0, x_1, \ldots, x_{I-1}) = x_0 \times x_1 \times \ldots \times x_{I-1}$

In [ ]:
def f(x: Shaped[Array, "I"]) -> Shaped[Array, "1"]:
    return np.prod(x, keepdims=True)


def jac(x: Shaped[Array, "I"]) -> Shaped[Array, "1 I"]:
    return 0 * Is[None, :]  # fill in


check(f, jac, in_shape=Is.shape)

### Problem 5: Repeat

$f(x_0) = [x_0, x_0,  x_0, \ldots  x_0]$

Hint: The function `dx` should return a scalar. It is the
derivative of $f(x_0)_o$, i.e. the o'th output.


In [ ]:
Is = np.arange(1)
O = 10
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "1"]) -> Shaped[Array, "O"]:
    return (x + (Os * 0 + 1))[:, 0]


def jac(x: Shaped[Array, "1"]) -> Shaped[Array, "O 1"]:
    return 0 * Os  # fill in


check(f, jac, in_shape=Is.shape)

### Problem 6: Repeat and Scale

$$f(x_0) = [x_0 \times 0/I, x_0 \times 1/I,  x_0 \times 2/I,  \ldots,  x_{0} \times (I-1)/I]$$

In [ ]:
Is = np.arange(1)
O = 10
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "1"]) -> Shaped[Array, "O"]:
    return x * (Os / O)[:, 0]


def jac(x: Shaped[Array, "1"]) -> Shaped[Array, "O 1"]:
    return 0 * Os  # fill in


check(f, jac, in_shape=Is.shape)

## Problem 7: Negation

$$f(x_0, x_1, \ldots) = [-x_0, -x_1, \ldots, -x_{I-1}]$$

In [ ]:
I = 10
O = 10
Is = np.arange(I)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"]) -> Shaped[Array, "O"]:
    return -x


def jac(x: Shaped[Array, "I"]) -> Shaped[Array, "O I"]:
    return 0 * (Os + Is)  # fill in


check(f, jac, in_shape=Is.shape)

## Problem 8: ReLU

$$f(x_0, x_1, \ldots) = [\text{relu}(x_0), \text{relu}(x_1), \ldots]$$

Recall

$$
\text{relu}(x) = \begin{cases}
0 & x < 0 \\
x & x >= 0
\end{cases}
$$

(Note: you can ignore the not of non-differentiability at 0.)

In [ ]:
I = 10
O = 10
Is = np.arange(I)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"]) -> Shaped[Array, "O"]:
    return x * (x > 0)


def jac(x: Shaped[Array, "I"]) -> Shaped[Array, "O I"]:
    return 0 * (Os + Is)  # fill in


check(f, jac, in_shape=Is.shape)

## Problem 9: Index

$$f(x_0, x_1, \ldots, x_{24}) = [x_{10}, x_{11}, \ldots, x_{24}]$$



In [ ]:
I = 25
O = 15
Is = np.arange(I)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"]) -> Shaped[Array, "O"]:
    return x[10:]


def jac(x: Shaped[Array, "I"]) -> Shaped[Array, "O I"]:
    return 0 * (Os + Is)  # fill in


check(f, jac, in_shape=Is.shape)

## Problem 10: Cumsum

$$f(x_0, x_1, \ldots) = [\sum^0_{i=0} x_{i}, \sum^1_{i=0} x_{i}, \sum^2_{i=0} x_{i}, \ldots, ] / 20$$



In [ ]:
I = 10
O = 10
Is = np.arange(I)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"]) -> Shaped[Array, "O"]:
    return np.cumsum(x) / 20


def jac(x: Shaped[Array, "I"]) -> Shaped[Array, "O I"]:
    return 0 * (Os + Is)  # fill in


check(f, jac, in_shape=Is.shape)

## Problem 11: Running Mean


$$f(x_0, x_1, \ldots)_o = \frac{\displaystyle \sum^o_{i=o-W} x_i}{W}$$

In [ ]:
I = 10
O = 8
Is = np.arange(I)
Os = np.arange(O)[:, None]
W = 3


def f(x: Shaped[Array, "I"]) -> Shaped[Array, "O"]:
    return np.convolve(x, np.ones(W) / W, mode="valid")


def jac(x: Shaped[Array, "I"]) -> Shaped[Array, "O I"]:
    return 0 * (Os + Is)  # fill in


check(f, jac, in_shape=Is.shape)

## Problem 12: Sort


$$f(x_0, x_1, \ldots) = \text{x's in sorted order}$$

You can use function like np.sort of this one.
(This one is a bit counterintuitive! Note that we are not asking you to differentiate the sorting function it self.)

In [ ]:
I = 10
O = 10
Is = np.arange(I)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"]) -> Shaped[Array, "O"]:
    return np.sort(x)


def jac(x: Shaped[Array, "I"]) -> Shaped[Array, "O I"]:
    return 0 * (Os + Is)  # fill in


check(f, jac, in_shape=Is.shape)

Next we move on to functions of two arguments. For these you will produce two derivatives: $$\frac{df(x, y)_o}{x_i}, \frac{df(x, y)_o}{y_j}$$. Everything else is the same.

## Problem 13: Elementwise mean

$$f(x, y)_o = (x_o + y_o) /2 $$

In [ ]:
I = 10
O = 10
Is = np.arange(I)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"], y: Shaped[Array, "I"]) -> Shaped[Array, "O"]:
    return (x + y) / 2


def jac(
    x: Shaped[Array, "I"], y: Shaped[Array, "I"]
) -> tuple[Shaped[Array, "O I"], Shaped[Array, "O I"]]:
    return 0 * (Os + Is), 0 * (Os + Is)  # fill in


check(f, jac, in_shape=Is.shape, in_shape2=Is.shape)

## Problem 14: Elementwise mul

$$f(x, y)_o = x_o * y_o $$

In [ ]:
I = 10
O = 10
Is = np.arange(I)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"], y: Shaped[Array, "I"]) -> Shaped[Array, "O"]:
    return x * y


def jac(
    x: Shaped[Array, "I"], y: Shaped[Array, "I"]
) -> tuple[Shaped[Array, "O I"], Shaped[Array, "O I"]]:
    return 0 * (Os + Is), 0 * (Os + Is)  # fill in


check(f, jac, in_shape=Is.shape, in_shape2=Is.shape)

## Problem 15: 1D Convolution

This is the standard 1D convolution used in deep learning. There is no wrap-around.

$$f(x, y)_o = \sum_{j=1}^K x_{o+j} * y_{j} / K $$

Note: This is probably the hardest one. The answer is short but tricky.

In [ ]:
I = 10
O = 6
W = 5
Is = np.arange(I)
Os = np.arange(O)[:, None]
Ws = np.arange(W)


def f(x: Shaped[Array, "I"], y: Shaped[Array, "W"]) -> Shaped[Array, "O"]:
    return np.convolve(x, y, mode="valid") / W


def jac(
    x: Shaped[Array, "I"], y: Shaped[Array, "W"]
) -> tuple[Shaped[Array, "O I"], Shaped[Array, "O W"]]:
    return 0 * (Is + Os), 0 * (Os + Ws)  # fill in


check(f, jac, in_shape=Is.shape, in_shape2=Ws.shape)

## Problem 15: View

Compute the identity function for all $o,p$. $y$ is ignored.

$$f(X)_{o, p} = X_{o, p}$$

In [ ]:
I = 4
J = 4
O = 16
Is = np.arange(I)[:, None]
Js = np.arange(J)
Os = np.arange(O)[:, None, None]


def f(x: Shaped[Array, "I J"]) -> Shaped[Array, "O"]:
    return x.reshape((O,))


def jac(x: Shaped[Array, "I J"]) -> Shaped[Array, "O I J"]:
    return 0 * Os + Is + Js  # fill in


check(f, jac, in_shape=(I, J))

## Problem 16: Transpose

Transpose row and columns

$$f(X, y)_{o, p} = X_{p, o}$$


In [ ]:
I = 4
J = 4
O = 4
P = 4
Is = np.arange(I)[:, None]
Js = np.arange(J)
Os = np.arange(O)[:, None, None]
Ps = np.arange(P)[:, None, None, None]


def f(x: Shaped[Array, "I J"]) -> Shaped[Array, "O P"]:
    return x.T


def jac(x: Shaped[Array, "I J"]) -> Shaped[Array, "O P I J"]:
    return 0 * (Os + Is + Ps + Js)  # fill in


check(f, jac, in_shape=(I, J))

## Problem 18: Mean Reduce

Compute the mean over rows

$$f(X, y)_{o, p} = \sum_{i} X_{i, p} / R$$


In [ ]:
I = 4
J = 4
O = 4
Is = np.arange(I)[:, None]
Js = np.arange(J)
Os = np.arange(O)[:, None, None]


def f(x: Shaped[Array, "I J"]) -> Shaped[Array, "O"]:
    return np.mean(x, axis=0)


def jac(x: Shaped[Array, "I J"]) -> Shaped[Array, "O I J"]:
    return 0 * (Is + Os + Js)  # fill in


check(f, jac, in_shape=(I, J))

## Problem 19: Matmul as vectors

Standard matrix multiplication

$$f(X, Y)_{o,p} = \sum_j X_{o, j} Y_{j,p}$$


In [ ]:
I = 4
J = 4
O = 4
Is = np.arange(I)
Js = np.arange(J)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"], y: Shaped[Array, "J"]) -> Shaped[Array, "O "]:
    return (x.reshape(2, 2) @ y.reshape(2, 2)).reshape(O)


def jac(
    x: Shaped[Array, "I"], y: Shaped[Array, "J"]
) -> tuple[Shaped[Array, "O I"], Shaped[Array, "O J"]]:
    return 0 * (Os == Is), 0 * (Os == Js)  # fill in


check(f, jac, in_shape=Is.shape, in_shape2=Js.shape)

## Problem 20: 2D Conv

Two Dimensional convolution

$$f(X, Y)_{o} = \sum_{dh}\sum_{dw} X_{o+dh*W+dw} Y_{dh,dw}$$



In [ ]:
from jax.scipy.signal import convolve2d

I = 16  # 4*4
J = 4  # 2*2
O = 16  # 4*4
Is = np.arange(I)
Js = np.arange(J)
Os = np.arange(O)[:, None]


def f(x: Shaped[Array, "I"], y: Shaped[Array, "J"]) -> Shaped[Array, "O"]:
    X = x.reshape(4, 4)
    Y = y.reshape(2, 2)

    return convolve2d(X, Y, mode="same").reshape(-1) / (2 * 2)


def jac(
    x: Shaped[Array, "I"], y: Shaped[Array, "J"]
) -> tuple[Shaped[Array, "O I"], Shaped[Array, "O J"]]:
    return 0 * (Os + Is), 0 * (Os + Js)  # fill in


check(f, jac, in_shape=Is.shape, in_shape2=Js.shape)